# FiGuard

[![PyPI](https://img.shields.io/pypi/v/figuard.svg)](https://pypi.org/project/figuard/)
[![GitHub](https://img.shields.io/badge/github-figuard%2Ffiguard--core-blue)](https://github.com/figuard/figuard-core)

[FiGuard](https://github.com/figuard/figuard-core) is a pre-flight spend authorization layer for AI agents. Before a tool call executes, FiGuard checks whether the agent is allowed to make that spend — and blocks it if not.

**The problem it solves:** LangChain agents can call tools in a loop. Without a control layer, a single stuck or runaway agent can spend thousands of dollars before anyone notices.

**How it works:** You create a budget (e.g. $500 for 24 hours). FiGuard intercepts every tool call via `FiGuardCallbackHandler`, asks the server "can this agent spend $X right now?", and raises `ToolException` on denial. The LLM receives the denial reason and can adapt its plan.

**No account required** — the examples below connect to a shared public sandbox automatically.

## Setup

In [ ]:
# @title Step 1 — Install and configure FiGuard
%pip install -qU figuard[langchain]

FiGuard connects to a shared public sandbox by default — no API key or account needed for these examples. For production, self-host FiGuard and set `FIGUARD_API_KEY` / `FIGUARD_BASE_URL`.

## Quickstart: stop a runaway agent in one line

`auto_guard_langchain` is the fastest path: one call creates a budget, wires the handler, and returns the same executor ready to run.

In [ ]:
from langchain.tools import tool
from langchain.agents import AgentExecutor, BaseSingleActionAgent
from langchain_core.agents import AgentFinish
from figuard.integrations.langchain import auto_guard_langchain


@tool
def check_price(destination: str) -> str:
    """Check travel prices. Args: destination (str)."""
    return f"Best price to {destination}: $299"


@tool
def book_service(service: str, cost: float) -> str:
    """Book a paid service. Args: service (str), cost (float USD)."""
    return f"Booked {service} for ${cost:.2f}"


# Minimal stub agent — swap for create_openai_tools_agent(ChatOpenAI(...), tools, prompt)
# No API key needed for this quickstart demo.
class _StubAgent(BaseSingleActionAgent):
    @property
    def input_keys(self):
        return ["input"]

    def plan(self, intermediate_steps, **kwargs):
        return AgentFinish({"output": "stub response"}, "stub")

    async def aplan(self, intermediate_steps, **kwargs):
        return self.plan(intermediate_steps, **kwargs)


executor = AgentExecutor(
    agent=_StubAgent(),
    tools=[check_price, book_service],
    handle_tool_error=True,
)

# One line: creates a $500 / 24h budget, wires FiGuardCallbackHandler
executor = auto_guard_langchain(executor, budget=500)
print("Executor is budget-protected ($500 / 24h).")
print(f"Handler: {type(executor.callbacks[0]).__name__} attached")
print("Swap _StubAgent for your real agent — see Examples 1–5 below for full setup.")


## Core concepts

FiGuard uses a **reserve-then-confirm** model:

1. `on_tool_start` → `authorize()` — checks the budget and reserves the requested amount
2. `on_tool_end` → `confirm_event()` — finalizes the spend with the actual cost
3. `on_tool_error` → `fail_event()` — releases the reservation if the tool fails

The reservation prevents two concurrent tools from both seeing "$200 available" and both getting approved when only $200 is left.

## Example 1: FiGuardCallbackHandler — basic setup

In [ ]:
from figuard import FiGuardClient
from figuard.integrations.langchain import FiGuardCallbackHandler

# Zero-config: connects to the shared sandbox automatically
client = FiGuardClient()

# Create a $500 budget valid for 24 hours
budget = client.create_budget(
    user_id="demo_user",
    total_limit=500.00,
    currency="USD",
    expires_in="24h",
)
print(f"Budget created: {budget.id}  available: ${budget.available_quantity:.2f}")

# Attach the handler to your AgentExecutor via the callbacks parameter:
#   executor = AgentExecutor(
#       agent=agent,
#       tools=tools,
#       handle_tool_error=True,   # required: sends denial reason to the LLM
#       callbacks=[handler],
#   )
handler = FiGuardCallbackHandler(
    client=client,
    session_token=budget.primary_token.session_token,
    agent_id="travel_agent",
)
print(f"Handler ready — agent_id: {handler.agent_id}")
print(f"Pass it to AgentExecutor(callbacks=[handler]) to pre-authorize every tool call.")


## Example 2: Runaway loop — agent tries to spend $900, budget is $500

This example drives the callback handler directly (no LLM required) to show exactly what happens when an agent exceeds its budget.

In [ ]:
from uuid import uuid4
from langchain_core.tools import ToolException
from figuard import FiGuardClient
from figuard.integrations.langchain import FiGuardCallbackHandler

client = FiGuardClient()
budget = client.create_budget(
    user_id="demo_user",
    total_limit=500.00,
    currency="USD",
    expires_in="24h",
)

handler = FiGuardCallbackHandler(
    client=client,
    session_token=budget.primary_token.session_token,
    agent_id="travel_agent",
)

# Simulated tool calls the agent wants to make:
calls = [
    ("book_flight",  300.00, "NYC to LAX"),
    ("book_hotel",   150.00, "Hotel in LA"),
    ("book_rental",  200.00, "Car rental"),  # This one exceeds the remaining $50
    ("book_dinner",   80.00, "Restaurant"),  # This one will also be denied
]

for tool_name, amount, description in calls:
    run_id = uuid4()
    try:
        handler.on_tool_start(
            {"name": tool_name},
            f'{{"amount": {amount}, "description": "{description}"}}',
            run_id=run_id,
        )
        # Tool executed — confirm with actual cost
        handler.on_tool_end(f"{description} confirmed", run_id=run_id)
        print(f"  ✓  {tool_name:<15} ${amount:>7.2f}   AUTHORIZED")
    except ToolException as e:
        print(f"  ✗  {tool_name:<15} ${amount:>7.2f}   {e}")

# Show final budget state
final = client.get_budget(budget.id)
print(f"\nBudget: spent=${final.quantity_spent:.2f}  available=${final.available_quantity:.2f}")

Expected output:
```
  ✓  book_flight      $300.00   AUTHORIZED
  ✓  book_hotel       $150.00   AUTHORIZED
  ✗  book_rental      $200.00   FiGuard DENIED: INSUFFICIENT_FUNDS — $50.00 remaining, $200.00 requested
  ✗  book_dinner       $80.00   FiGuard DENIED: INSUFFICIENT_FUNDS — $50.00 remaining, $80.00 requested

Budget: spent=$450.00  available=$50.00
```

With `handle_tool_error=True` on the `AgentExecutor`, the LLM receives the denial reason as the tool result and can adapt — for example, suggesting a cheaper hotel or asking the user to increase the budget.

## Example 3: Category-based budgets (allocations)

Ring-fence spend by category. A hotel booking cannot consume the flights allocation even if the total budget has headroom.

In [ ]:
from figuard import FiGuardClient
from figuard.integrations.langchain import FiGuardCallbackHandler
from langchain_core.tools import ToolException
from uuid import uuid4

client = FiGuardClient()
budget = client.create_budget(
    user_id="demo_user",
    total_limit=500.00,
    currency="USD",
    expires_in="24h",
    allocations=[
        {"category": "flights", "limit": 300.00, "enforcementMode": "STRICT"},
        {"category": "hotels",  "limit": 200.00, "enforcementMode": "STRICT"},
    ],
)

handler = FiGuardCallbackHandler(
    client=client,
    session_token=budget.primary_token.session_token,
    agent_id="travel_agent",
    tool_category_map={
        "book_flight": "flights",
        "book_hotel":  "hotels",
    },
)

run_id = uuid4()
try:
    # This hotel booking exceeds the hotels allocation ($250 > $200)
    handler.on_tool_start(
        {"name": "book_hotel"},
        '{"amount": 250.0, "hotel": "Grand Hyatt"}',
        run_id=run_id,
    )
except ToolException as e:
    print(f"Hotel denied: {e}")
    # Total budget still has $500 available — but hotels allocation is only $200
    # The LLM will be told to find a cheaper option

run_id2 = uuid4()
handler.on_tool_start(
    {"name": "book_hotel"},
    '{"amount": 180.0, "hotel": "Holiday Inn"}',
    run_id=run_id2,
)
handler.on_tool_end("Booked", run_id=run_id2)
print("Cheaper hotel: AUTHORIZED")

## Example 4: FiGuardToolGuard — per-tool hard enforcement

`FiGuardToolGuard` wraps a single tool's `_run` method in-place. Unlike the callback handler, it enforces regardless of `AgentExecutor` configuration and returns a denial string instead of raising an exception — the LLM receives it as a normal tool result.

In [ ]:
from langchain_core.tools import BaseTool
from pydantic import BaseModel
from figuard import FiGuardClient
from figuard.integrations.langchain import FiGuardToolGuard

class BookFlightInput(BaseModel):
    destination: str
    price: float

class BookFlightTool(BaseTool):
    name: str = "book_flight"
    description: str = "Book a flight. Args: destination (str), price (float USD)."
    args_schema: type[BaseModel] = BookFlightInput

    def _run(self, destination: str, price: float) -> str:
        return f"Flight to {destination} booked for ${price}"

client = FiGuardClient()
budget = client.create_budget(
    user_id="demo_user",
    total_limit=200.00,
    currency="USD",
    expires_in="24h",
)

book_flight = BookFlightTool()

FiGuardToolGuard(
    tool=book_flight,
    client=client,
    session_token=budget.primary_token.session_token,
    amount_key="price",
    agent_id="flight_agent",
)

# Authorized: $150 fits within $200 budget
result = book_flight._run(destination="NYC", price=150.0)
print(f"Result 1: {result}")

# Denied: $100 would exceed remaining $50
result = book_flight._run(destination="LAX", price=100.0)
print(f"Result 2: {result}")

## Example 5: ignore_tools — skip read-only operations

Pass `ignore_tools` to skip FiGuard authorization for tools that don't spend money (search, lookup, read operations).

In [ ]:
from figuard import FiGuardClient
from figuard.integrations.langchain import FiGuardCallbackHandler

client = FiGuardClient()
budget = client.create_budget(
    user_id="demo_user",
    total_limit=500.00,
    currency="USD",
    expires_in="24h",
)

handler = FiGuardCallbackHandler(
    client=client,
    session_token=budget.primary_token.session_token,
    agent_id="research_agent",
    ignore_tools={"search_web", "read_file", "get_weather"},  # skip these
)

# executor = AgentExecutor(agent=agent, tools=tools, handle_tool_error=True, callbacks=[handler])
print("Handler configured: search_web, read_file, get_weather will bypass FiGuard")

## Spend audit — what happened during the run

In [ ]:
# After your agent run completes, inspect the ledger.
# This cell uses `client` and `budget` from Example 2 — run that cell first.
# To inspect a different budget, replace budget.id with any budget ID.

page = client.get_ledger(budget.id, page=0, size=20)
print(f"Total events: {page.total_elements}")
print()
for event in page.events:
    status = "\u2713" if event.decision == "CONFIRMED" else "\u2717"
    amount = event.confirmed_quantity or event.requested_quantity
    print(f"  {status}  {event.decision:<12}  ${amount:>7.2f}  {event.agent_id}")


## Handler vs. Tool Guard — when to use which

| | `FiGuardCallbackHandler` | `FiGuardToolGuard` |
|---|---|---|
| Attach to | `AgentExecutor` callbacks | Individual tool (`_run` patched in-place) |
| Denial behavior | Raises `ToolException` → LLM receives reason | Returns denial string → LLM receives reason |
| Per-tool categories | Via `tool_category_map` dict | Via `category` param on each guard |
| LangGraph support | Yes (wire via `config`) | Yes (tool is unchanged) |
| Best for | Most single-agent setups | Per-tool control; different budgets per tool |

`handle_tool_error=True` is required on `AgentExecutor` when using `FiGuardCallbackHandler`, so the LLM receives the denial reason instead of the run crashing.

## API reference

- [FiGuard GitHub](https://github.com/figuard/figuard-core)
- [Python SDK docs](https://github.com/figuard/figuard-core/tree/main/sdk/python)
- [LangChain integration guide](https://github.com/figuard/figuard-core/blob/main/docs/integrations/langchain.md)
- [LangGraph integration guide](https://github.com/figuard/figuard-core/blob/main/docs/integrations/langgraph.md)
- [Self-hosting](https://github.com/figuard/figuard-core/blob/main/docs/self-hosting.md)